# Lab 4 — Cluster Metrics

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

Evaluate K-Means segmentation quality with three standard internal metrics.

**Dataset:** `data/nyse/nyse_stocks.csv` (500 rows, 25 symbols)

## Setup baseline model

In [ ]:
%matplotlib inline

from pathlib import Path

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif (GH_ROOT.parent / "notebooks").is_dir() and (GH_ROOT.parents[1] / "requirements-student.txt").is_file():
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "requirements-student.txt").is_file():
            GH_ROOT = parent
            break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score
from sklearn.preprocessing import StandardScaler

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])
k = 4
labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
print(f"symbols: {len(features)}, k: {k}")


## Compute silhouette, DB, and CH

In [ ]:
sil = silhouette_score(X_scaled, labels)
db = davies_bouldin_score(X_scaled, labels)
ch = calinski_harabasz_score(X_scaled, labels)

print("Lab 4 — Cluster metrics")
print(f"k: {k}")
print(f"silhouette score: {sil:.4f}  (higher is better, max 1)")
print(f"Davies-Bouldin index: {db:.4f}  (lower is better)")
print(f"Calinski-Harabasz score: {ch:.4f}  (higher is better)")

metrics_df = pd.DataFrame({
    "metric": ["silhouette", "davies_bouldin", "calinski_harabasz"],
    "value": [sil, db, ch],
    "direction": ["higher better", "lower better", "higher better"],
})
display(metrics_df.round(4))


## Per-sample silhouette details

In [ ]:
from sklearn.metrics import silhouette_samples

sample_scores = silhouette_samples(X_scaled, labels)
features = features.copy()
features["cluster"] = labels
features["silhouette"] = sample_scores.round(3)

display(
    features[["symbol", "cluster", "silhouette"]]
    .sort_values("silhouette")
    .head(5)
)


## Compare metric values for k=3 vs k=4

In [ ]:
rows = []
for k_try in [3, 4]:
    lbl = KMeans(n_clusters=k_try, random_state=42, n_init=10).fit_predict(X_scaled)
    rows.append({
        "k": k_try,
        "silhouette": silhouette_score(X_scaled, lbl),
        "davies_bouldin": davies_bouldin_score(X_scaled, lbl),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, lbl),
    })

compare_k = pd.DataFrame(rows)
display(compare_k.round(4))


## Visualize normalized metrics

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = metrics_df.copy()
plot_df["display_value"] = plot_df["value"] / plot_df["value"].max()
sns.barplot(data=plot_df, x="metric", y="display_value", ax=ax, palette="Set2")
ax.set_ylabel("normalized value (for display)")
ax.set_title(f"Cluster metrics (k={k}) — check direction column for interpretation")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### Metric interpretation prompt 1

Show metrics table sorted by absolute value.

In [ ]:
display(metrics_df.sort_values('value', ascending=False).round(4))

### Metric interpretation prompt 2

Compute silhouette summary by cluster.

In [ ]:
display(features.groupby('cluster')['silhouette'].agg(['mean','min','max']).round(3))

### Metric interpretation prompt 3

Count negative silhouette points.

In [ ]:
print(int((features['silhouette'] < 0).sum()))

### Metric interpretation prompt 4

List symbols with lowest silhouette.

In [ ]:
display(features.nsmallest(5, 'silhouette')[['symbol','cluster','silhouette']])

### Metric interpretation prompt 5

List symbols with highest silhouette.

In [ ]:
display(features.nlargest(5, 'silhouette')[['symbol','cluster','silhouette']])

### Metric interpretation prompt 6

Compare k=3 metric row.

In [ ]:
display(compare_k.loc[compare_k['k']==3].round(4))

### Metric interpretation prompt 7

Compare k=4 metric row.

In [ ]:
display(compare_k.loc[compare_k['k']==4].round(4))

### Metric interpretation prompt 8

Compute custom score: silhouette / DB.

In [ ]:
print(round(float(sil / db), 4))

### Metric interpretation prompt 9

Compute ratio CH/DB.

In [ ]:
print(round(float(ch / db), 4))

### Metric interpretation prompt 10

Display per-cluster counts.

In [ ]:
print(features['cluster'].value_counts().sort_index().to_dict())

### Metric interpretation prompt 11

Histogram of silhouette scores.

In [ ]:
ax = features['silhouette'].plot(kind='hist', bins=10, figsize=(5,3)); ax.set_title('Silhouette score distribution');

### Metric interpretation prompt 12

Boxplot silhouette by cluster.

In [ ]:
ax = features.boxplot(column='silhouette', by='cluster', figsize=(6,3));

### Metric interpretation prompt 13

Summarize interpretation sentence.

In [ ]:
print('k=4 provides moderate separation with some overlap and one weaker cluster boundary.')

### Metric interpretation prompt 14

Check metric directions table.

In [ ]:
print(metrics_df[['metric','direction']].to_dict(orient='records'))

### Metric interpretation prompt 15

Create compact dict snapshot.

In [ ]:
print({'k': k, 'sil': round(sil,4), 'db': round(db,4), 'ch': round(ch,4)})

### Metric interpretation prompt 16

Difference between k=3 and k=4 silhouette.

In [ ]:
print(round(float(compare_k.loc[compare_k['k']==4, 'silhouette'].iloc[0] - compare_k.loc[compare_k['k']==3, 'silhouette'].iloc[0]), 4))

### Metric interpretation prompt 17

Difference between k=3 and k=4 DB.

In [ ]:
print(round(float(compare_k.loc[compare_k['k']==4, 'davies_bouldin'].iloc[0] - compare_k.loc[compare_k['k']==3, 'davies_bouldin'].iloc[0]), 4))

### Metric interpretation prompt 18

Difference between k=3 and k=4 CH.

In [ ]:
print(round(float(compare_k.loc[compare_k['k']==4, 'calinski_harabasz'].iloc[0] - compare_k.loc[compare_k['k']==3, 'calinski_harabasz'].iloc[0]), 4))

### Metric interpretation prompt 19

Bridge to visualization lab.

In [ ]:
print('Next notebook visualizes K-Means and DBSCAN outputs side-by-side.')

### Metric interpretation prompt 20

Confirm expected symbol count.

In [ ]:
print(len(features), features['symbol'].nunique())

### Lab 4 quick recap 1

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 1: completed")

### Lab 4 quick recap 2

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 2: completed")

### Lab 4 quick recap 3

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 3: completed")

### Lab 4 quick recap 4

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 4: completed")

### Lab 4 quick recap 5

Pause and summarize one takeaway from the previous cell before moving on.

In [ ]:
print("Lab 4 recap step 5: completed")

## Final checkpoint

In [ ]:
assert k == 4
assert abs(sil - 0.2414) < 0.05
assert abs(db - 1.0659) < 0.1
assert abs(ch - 8.2627) < 1.0
print("Numbers match — you're good.")



## Reflection

Which metric would you prioritize for dense, noisy financial features?